NLP Coursework Project
Replicating the text-embedding stage from:
"Chronologically Consistent Large Language Models" (He et al., 2025)

Goal of this notebook section:
1) Download/load FNSPID from Hugging Face
2) Aggregate headlines by stock and day
3) Load a ChronoBERT checkpoint
4) Build one embedding per (date, stock_symbol)
5) Save embeddings for later return-prediction experiments

In [ ]:
import os
from dataclasses import dataclass
from typing import Literal
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import list_repo_files, hf_hub_download
import zipfile
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt

In [ ]:
# Pick a writable base directory
BASE_DIR = os.path.expanduser("~/nlp_coursework")
os.makedirs(BASE_DIR, exist_ok=True)
DATASET_REPO = "Zihan1004/FNSPID"
NEWS_FILENAME = "Stock_news/All_external.csv"

local_path = hf_hub_download(
    repo_id=DATASET_REPO,
    repo_type="dataset",
    filename=NEWS_FILENAME,
)

@dataclass
class Config:
    # Input data
    news_csv_path: str = local_path   # path returned by hf_hub_download

    # Output location
    output_dir: str = os.path.join(BASE_DIR, "outputs")

    # Data columns
    date_col: str = "Date"
    symbol_col: str = "Stock_symbol"
    text_col: str = "Article_title"

    # Date window
    start_year: int = 2009 #everything pre 2009 is either not in english or missing stock tickers
    end_year: int = 2020

    # Debugging controls
    debug: bool = True
    debug_rows_per_year: int = 40000   # only used if debug=True

    # Chunked CSV reading
    chunksize: int = 200_000

    # Embedding extraction
    max_length: int = 256
    batch_size: int = 16
    hidden_state_strategy: Literal["last", "first", "all"] = "last"

    # Runtime
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Config()
RUN_SMOKE_TESTS = False

os.makedirs(cfg.output_dir, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Output directory:", cfg.output_dir)
print("Debug mode:", cfg.debug)
if cfg.debug:
    print("Rows per year in debug mode:", cfg.debug_rows_per_year)

cfg

Baseline models to compare - note that the ChronoBERT rolling models will be generated from the date later

In [ ]:
MODEL_REGISTRY = {
    # Leaked baseline within the same model family
    "chronobert_20241231": "manelalab/chrono-bert-v1-20241231",

    # Non-chronological finance model
    "finbert": "ProsusAI/finbert",

    # Non-chronological modern encoder
    "modernbert_base": "answerdotai/ModernBERT-base",
}
MODEL_REGISTRY

Function to generate the correct ChronoBERT model from the year

In [ ]:
def chrono_model_for_year(eval_year: int) -> str:
    """
    For news in year Y, use the model checkpoint from Dec 31 of year Y-1.
    Example:
      eval_year=2020 -> manelalab/chrono-bert-v1-20191231
    """
    checkpoint_year = eval_year - 1
    if checkpoint_year < 1999:
        raise ValueError(f"No chronologically safe checkpoint available for eval_year={eval_year}")
    return f"manelalab/chrono-bert-v1-{checkpoint_year}1231"

#test the function
for y in [2000, 2008, 2009, 2020]:
    print(y, "->", chrono_model_for_year(y))

Check which files are available to download from fnspid - downloaded All_external

Loading the 5k rows per year for debug as full dataset is too big
It seems like everything pre 2009 is missing ticker symbols and is in Russian so we need to remove all of that

In [ ]:
start_date = pd.to_datetime(f"{cfg.start_year}-01-01").date()
end_date   = pd.to_datetime(f"{cfg.end_year}-12-31").date()

usecols = [cfg.date_col, cfg.symbol_col, cfg.text_col]

total_rows_seen = 0
filtered_chunks = []

for chunk in tqdm(
    pd.read_csv(
        cfg.news_csv_path,
        usecols=usecols,
        chunksize=cfg.chunksize,
        dtype={cfg.symbol_col: str},
        low_memory=False,
    ),
    desc="Reading / cleaning news chunks"
):
    total_rows_seen += len(chunk)

    # Parse and standardize
    datetime_series = pd.to_datetime(
        chunk[cfg.date_col],
        format="%Y-%m-%d %H:%M:%S UTC",
        errors="coerce",
        utc=True
    )
    eastern_times = datetime_series.dt.tz_convert("US/Eastern")
    chunk[cfg.date_col] = (eastern_times + pd.Timedelta(hours=8)).dt.date

    chunk[cfg.symbol_col] = chunk[cfg.symbol_col].fillna("").astype(str)
    chunk[cfg.text_col] = chunk[cfg.text_col].fillna("").astype(str)

    # Drop unusable rows
    chunk = chunk.dropna(subset=[cfg.date_col])
    chunk = chunk[chunk[cfg.text_col].str.strip() != ""]
    chunk = chunk[chunk[cfg.symbol_col].str.lower() != "nan"]
    chunk = chunk[chunk[cfg.symbol_col].str.strip() != ""]

    # Keep target date window
    chunk = chunk[
        (chunk[cfg.date_col] >= start_date) &
        (chunk[cfg.date_col] <= end_date)
    ].copy()

    if len(chunk) > 0:
        filtered_chunks.append(chunk)

news_df_full = pd.concat(filtered_chunks, ignore_index=True)
news_df_full["year"] = pd.to_datetime(news_df_full[cfg.date_col]).dt.year

news_df_full = (
    news_df_full
    .sort_values([cfg.date_col, cfg.symbol_col])
    .reset_index(drop=True)
)

print("Total raw rows seen:", total_rows_seen)
print("Filtered article rows:", len(news_df_full))
print("Year range:", news_df_full["year"].min(), "to", news_df_full["year"].max())
print("\nArticle rows per year:")
print(news_df_full["year"].value_counts().sort_index())

news_df_full.head()

In [ ]:
def aggregate_headlines_by_stock_day(df, date_col, symbol_col, text_col):
    grouped = (
        df.groupby([date_col, symbol_col])[text_col]
        .apply(lambda s: " ".join(x.strip() for x in s if x.strip()))
        .reset_index()
        .rename(columns={text_col: "aggregated_text"})
    )
    return grouped

stock_day_df_full = aggregate_headlines_by_stock_day(
    news_df_full,
    date_col=cfg.date_col,
    symbol_col=cfg.symbol_col,
    text_col=cfg.text_col,
)

stock_day_df_full["year"] = pd.to_datetime(stock_day_df_full[cfg.date_col]).dt.year
stock_day_df_full = (
    stock_day_df_full
    .sort_values([cfg.date_col, cfg.symbol_col])
    .reset_index(drop=True)
)

print(stock_day_df_full.shape)
print("Stock-day rows per year:")
print(stock_day_df_full["year"].value_counts().sort_index())
stock_day_df_full.head()

In [ ]:
if cfg.debug:
    stock_day_df = (
        stock_day_df_full.groupby("year", group_keys=False)
        .apply(lambda x: x.sample(min(len(x), cfg.debug_rows_per_year), random_state=42))
        .reset_index(drop=True)
    )
    print(f"DEBUG MODE: sampled up to {cfg.debug_rows_per_year:,} stock-day rows per year")
else:
    stock_day_df = stock_day_df_full.copy()
    print("FULL MODE: using all stock-day rows")

stock_day_df = (
    stock_day_df
    .sort_values([cfg.date_col, cfg.symbol_col])
    .reset_index(drop=True)
)

print(stock_day_df.shape)
print("Working stock-day rows per year:")
print(stock_day_df["year"].value_counts().sort_index())
stock_day_df.head()

In [ ]:
def load_model_and_tokenizer(model_name, device):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name, output_hidden_states=True)
    model.to(device)
    model.eval()
    return tokenizer, model

In [ ]:
def mean_pool_masked(hidden: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).expand(hidden.size()).float()
    summed = (hidden * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def get_sequence_embedding(outputs, attention_mask, strategy="last"):
    hidden_states = outputs.hidden_states

    if strategy == "last":
        token_hidden = hidden_states[-1]
    elif strategy == "first":
        token_hidden = hidden_states[1]
    elif strategy == "all":
        stacked = torch.stack(hidden_states[1:], dim=0)   # [L, B, T, H]
        token_hidden = stacked.mean(dim=0)                # [B, T, H]
    else:
        raise ValueError("strategy must be one of: 'last', 'first', 'all'")

    return mean_pool_masked(token_hidden, attention_mask)

@torch.no_grad()
def embed_texts(
    texts,
    tokenizer,
    model,
    max_length=256,
    batch_size=16,
    strategy="last",
    device="cpu",
):
    all_embeddings = []

    for start in tqdm(range(0, len(texts), batch_size), desc="Embedding texts"):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}
        outputs = model(**encoded)

        seq_emb = get_sequence_embedding(
            outputs=outputs,
            attention_mask=encoded["attention_mask"],
            strategy=strategy,
        )

        all_embeddings.append(seq_emb.cpu().numpy())

    return np.vstack(all_embeddings)

In [ ]:
def build_embeddings_for_model(
    stock_day_df,
    model_name,
    date_col,
    symbol_col,
    text_col="aggregated_text",
    max_length=256,
    batch_size=16,
    strategy="last",
    device="cpu",
):
    tokenizer, model = load_model_and_tokenizer(model_name, device)

    texts = stock_day_df[text_col].tolist()

    embeddings = embed_texts(
        texts=texts,
        tokenizer=tokenizer,
        model=model,
        max_length=max_length,
        batch_size=batch_size,
        strategy=strategy,
        device=device,
    )

    emb_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]
    emb_df = pd.DataFrame(embeddings, columns=emb_cols)

    out = pd.concat(
        [
            stock_day_df[[date_col, symbol_col]].reset_index(drop=True),
            emb_df,
        ],
        axis=1,
    )

    # Explicit cleanup
    del tokenizer
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out

In [ ]:
def build_rolling_chronobert_embeddings(
    stock_day_df,
    date_col,
    symbol_col,
    text_col="aggregated_text",
    max_length=256,
    batch_size=16,
    strategy="last",
    device="cpu",
    output_dir=None,
):
    yearly_outputs = []

    for year in sorted(stock_day_df["year"].unique()):
        model_name = chrono_model_for_year(year)
        year_df = stock_day_df[stock_day_df["year"] == year].copy()

        print(f"\n=== Embedding year {year} with {model_name} ===")
        print(f"Rows: {len(year_df)}")

        emb_df = build_embeddings_for_model(
            stock_day_df=year_df,
            model_name=model_name,
            date_col=date_col,
            symbol_col=symbol_col,
            text_col=text_col,
            max_length=max_length,
            batch_size=batch_size,
            strategy=strategy,
            device=device,
        )

        emb_df["eval_year"] = year
        emb_df["model_name"] = model_name
        emb_df["model_key"] = "rolling_chronobert"

        yearly_outputs.append(emb_df)

        if output_dir is not None:
            year_path = os.path.join(output_dir, f"embeddings_rolling_chronobert_{year}.parquet")
            emb_df.to_parquet(year_path, index=False)

            combined_path = os.path.join(output_dir, "embeddings_rolling_chronobert.parquet")
            pd.concat(yearly_outputs, ignore_index=True).to_parquet(combined_path, index=False)

            print(f"Saved yearly checkpoint: {year_path}")
            print(f"Updated combined rolling checkpoint: {combined_path}")

    return pd.concat(yearly_outputs, ignore_index=True)

In [ ]:
def build_fixed_model_embeddings(
    stock_day_df,
    model_name,
    model_key,
    date_col,
    symbol_col,
    text_col="aggregated_text",
    max_length=256,
    batch_size=16,
    strategy="last",
    device="cpu",
):
    print(f"\n=== Embedding all rows with {model_key}: {model_name} ===")
    print(f"Rows: {len(stock_day_df)}")

    emb_df = build_embeddings_for_model(
        stock_day_df=stock_day_df,
        model_name=model_name,
        date_col=date_col,
        symbol_col=symbol_col,
        text_col=text_col,
        max_length=max_length,
        batch_size=batch_size,
        strategy=strategy,
        device=device,
    )

    emb_df["eval_year"] = stock_day_df["year"].values
    emb_df["model_name"] = model_name
    emb_df["model_key"] = model_key
    return emb_df

Create rolling embeddings for each chronoBERT model - each one has embeddings for just the next year

In [ ]:
# rolling_chrono_embeddings = build_rolling_chronobert_embeddings(
#     stock_day_df=stock_day_df,
#     date_col=cfg.date_col,
#     symbol_col=cfg.symbol_col,
#     text_col="aggregated_text",
#     max_length=cfg.max_length,
#     batch_size=cfg.batch_size,
#     strategy=cfg.hidden_state_strategy,
#     device=cfg.device,
#     output_dir=cfg.output_dir,
# )

Create embeddings for the 3 full models (chronoBERT 2024, finBERT, modernBERT)

In [ ]:
# baseline_paths = {}

# for model_key, model_name in MODEL_REGISTRY.items():
#     emb_df = build_fixed_model_embeddings(
#         stock_day_df=stock_day_df,
#         model_name=model_name,
#         model_key=model_key,
#         date_col=cfg.date_col,
#         symbol_col=cfg.symbol_col,
#         text_col="aggregated_text",
#         max_length=cfg.max_length,
#         batch_size=cfg.batch_size,
#         strategy=cfg.hidden_state_strategy,
#         device=cfg.device,
#     )

#     output_path = os.path.join(cfg.output_dir, f"embeddings_{model_key}.parquet")
#     emb_df.to_parquet(output_path, index=False)

#     baseline_paths[model_key] = output_path
#     print(f"Saved {model_key} -> {output_path}")

Check whats in the full_history price files - looks like we have 1 CSV per ticker. Need to just download the tickers that are relevant for us

In [ ]:
price_zip_path = hf_hub_download(
    repo_id="Zihan1004/FNSPID",
    repo_type="dataset",
    filename="Stock_price/full_history.zip",
)

print(price_zip_path)

In [ ]:
with zipfile.ZipFile(price_zip_path, "r") as z:
    zip_members = z.namelist()

print(f"Number of files inside zip: {len(zip_members)}")
for name in zip_members[:50]:
    print(name)

In [ ]:
embedding_paths = {
    "rolling_chronobert": os.path.join(cfg.output_dir, "embeddings_rolling_chronobert.parquet"),
    "chronobert_20241231": os.path.join(cfg.output_dir, "embeddings_chronobert_20241231.parquet"),
    "finbert": os.path.join(cfg.output_dir, "embeddings_finbert.parquet"),
    "modernbert_base": os.path.join(cfg.output_dir, "embeddings_modernbert_base.parquet"),
}

embedding_tables = {}

for model_name, path in embedding_paths.items():
    print(model_name, "->", path, "exists?", os.path.exists(path))
    embedding_tables[model_name] = pd.read_parquet(path)
    print(model_name, embedding_tables[model_name].shape)

In [ ]:


SUBSAMPLE_FROM_SAVED = False
saved_rows_per_year = 5000   # choose x here
subsample_random_state = 40

if SUBSAMPLE_FROM_SAVED:
    reference_df = embedding_tables["rolling_chronobert"].copy()
    reference_df["year"] = pd.to_datetime(reference_df["Date"]).dt.year

    sampled_keys = (
        reference_df
        .groupby("year", group_keys=False)
        .apply(lambda x: x.sample(min(len(x), saved_rows_per_year), random_state=subsample_random_state))
        .reset_index(drop=True)[["Date", "Stock_symbol"]]
        .drop_duplicates()
    )

    print(f"Subsampling saved embeddings to up to {saved_rows_per_year:,} rows per year")
    print("Number of sampled stock-day keys:", len(sampled_keys))

    sampled_embedding_tables = {}
    for model_name, emb_df in embedding_tables.items():
        tmp = emb_df.merge(sampled_keys, on=["Date", "Stock_symbol"], how="inner").copy()
        sampled_embedding_tables[model_name] = tmp

        tmp["year"] = pd.to_datetime(tmp["Date"]).dt.year
        print(f"\n{model_name}:")
        print(tmp.shape)
        print(tmp["year"].value_counts().sort_index())

    embedding_tables = sampled_embedding_tables
else:
    print("Using full saved embedding tables with no further subsampling")
    print(embedding_tables['finbert'].shape)

In [ ]:
# Build ticker universe from all embedding tables
ticker_universe = set()
for emb_df in embedding_tables.values():
    ticker_universe.update(emb_df["Stock_symbol"].astype(str).unique())

print("Number of tickers in embedding universe:", len(ticker_universe))
# print(list(sorted(ticker_universe))[:20])

price_csv_members = [
    m for m in zip_members
    if m.startswith("full_history/")
    and m.endswith(".csv")
    and "__MACOSX" not in m
]

print("Number of real price CSV files:", len(price_csv_members))
print(price_csv_members[:10])

In [ ]:
available_price_tickers = {
    m.split("/")[-1].replace(".csv", "") for m in price_csv_members
}

matched_tickers = sorted(ticker_universe & available_price_tickers)
missing_tickers = sorted(ticker_universe - available_price_tickers)

print("Matched tickers:", len(matched_tickers))
print("Missing tickers:", len(missing_tickers))
print("First 20 matched:", matched_tickers[:20])
print("First 20 missing:", missing_tickers[:20])

matched_ticker_set = set(matched_tickers)

rolling_diag_df = embedding_tables["rolling_chronobert"]

usable_embedding_rows = rolling_diag_df[
    rolling_diag_df["Stock_symbol"].isin(matched_ticker_set)
].copy()

dropped_embedding_rows = rolling_diag_df[
    ~rolling_diag_df["Stock_symbol"].isin(matched_ticker_set)
].copy()

print("Usable embedding rows:", len(usable_embedding_rows))
print("Dropped embedding rows:", len(dropped_embedding_rows))
print("Fraction dropped:", len(dropped_embedding_rows) / len(rolling_diag_df))

In [ ]:
sample_ticker = 'A'
sample_member = f"full_history/{sample_ticker}.csv"

with zipfile.ZipFile(price_zip_path, "r") as z:
    with z.open(sample_member) as f:
        sample_price_df = pd.read_csv(f)

print("Sample ticker:", sample_ticker)
print(sample_price_df.columns.tolist())
sample_price_df.tail()

In [ ]:
#download all the relevant tickers and stack into 1 df
price_frames = []

with zipfile.ZipFile(price_zip_path, "r") as z:
    for ticker in tqdm(matched_tickers, desc="Loading matched ticker price files"):
        member = f"full_history/{ticker}.csv"
        with z.open(member) as f:
            tmp = pd.read_csv(f)
            tmp["Stock_symbol"] = ticker
            price_frames.append(tmp)

price_df = pd.concat(price_frames, ignore_index=True)

print(price_df.shape)
price_df.head()

In [ ]:
#standardise
PRICE_DATE_COL = "date"
PRICE_SYMBOL_COL = "Stock_symbol"
PRICE_CLOSE_COL = "adj close"
price_df = price_df.copy()

price_df[PRICE_DATE_COL] = pd.to_datetime(price_df[PRICE_DATE_COL], errors="coerce").dt.date
price_df[PRICE_SYMBOL_COL] = price_df[PRICE_SYMBOL_COL].astype(str)

price_df = price_df.dropna(subset=[PRICE_DATE_COL, PRICE_CLOSE_COL])

# Keep only the date window you care about
start_date = pd.to_datetime(f"{cfg.start_year}-01-01").date()
end_date   = pd.to_datetime(f"{cfg.end_year}-12-31").date()

price_df = price_df[
    (price_df[PRICE_DATE_COL] >= start_date) &
    (price_df[PRICE_DATE_COL] <= end_date)
].copy()

# Sort ascending by ticker and date so shift(-1) means next trading day
price_df = price_df.sort_values([PRICE_SYMBOL_COL, PRICE_DATE_COL]).reset_index(drop=True)

print(price_df.shape)
price_df.head()

#check
sample_check = price_df[price_df["Stock_symbol"] == "AAPL"].head(10)
sample_check

In [ ]:
price_df["dollar_volume"] = price_df["adj close"] * price_df["volume"]

price_df["adv20"] = (
    price_df.groupby("Stock_symbol")["dollar_volume"]
    .transform(lambda s: s.rolling(20, min_periods=20).mean())
)

price_df["history_n"] = price_df.groupby("Stock_symbol").cumcount() + 1

eligible_df = price_df[
    (price_df["adj close"] >= 5) &
    (price_df["adv20"] >= 1_000_000) &
    (price_df["history_n"] >= 60)
][["date", "Stock_symbol"]].copy()

#eligible_df = price_df[["date", "Stock_symbol"]].copy()

eligible_df = eligible_df.rename(columns={"date": "Date"})

print("Eligible rows:", len(eligible_df))
print("Eligible tickers:", eligible_df["Stock_symbol"].nunique())
eligible_df.head()

In [ ]:
#compute next day returns

price_df["ret_next_day"] = (
    price_df.groupby(PRICE_SYMBOL_COL)[PRICE_CLOSE_COL]
    .shift(-1) / price_df[PRICE_CLOSE_COL] - 1.0
)

returns_df = price_df[[PRICE_DATE_COL, PRICE_SYMBOL_COL, "ret_next_day"]].copy()
returns_df = returns_df.rename(columns={
    PRICE_DATE_COL: "Date",
    PRICE_SYMBOL_COL: "Stock_symbol",
})

print(returns_df.shape)
returns_df.head()


In [ ]:
#function to fit the ridge regression

def fit_monthly_ridge_models(df, embedding_cols, target_col="ret_next_day", alpha=1.0):
    monthly_models = {}

    for month, month_df in sorted(df.groupby("month")):
        X_m = month_df[embedding_cols].values
        y_m = month_df[target_col].values

        if len(month_df) < 10:
            continue

        model = Ridge(alpha=alpha, fit_intercept=True)
        model.fit(X_m, y_m)

        monthly_models[month] = {
            "intercept": model.intercept_,
            "coef": model.coef_.copy(),
            "n_obs": len(month_df),
        }

    return monthly_models

In [ ]:
def build_oos_predictions(df, embedding_cols, monthly_models, target_col="ret_next_day"):
    months = sorted(df["month"].unique())
    out = []

    for month in months:
        past_months = [m for m in monthly_models.keys() if m < month]
        if len(past_months) == 0:
            continue

        avg_intercept = np.mean([monthly_models[m]["intercept"] for m in past_months])
        avg_coef = np.mean([monthly_models[m]["coef"] for m in past_months], axis=0)

        month_df = df[df["month"] == month].copy()
        X_m = month_df[embedding_cols].values

        month_df["pred_ret"] = avg_intercept + X_m @ avg_coef
        out.append(month_df)

    return pd.concat(out, ignore_index=True)

In [ ]:
def assign_daily_buckets(df, pred_col="pred_ret", n_buckets=10):
    """
    For each day:
    - sort stocks by predicted return
    - assign bucket 1,...,n_buckets
    - if fewer than n_buckets stocks that day, assign NaN
    """
    out = []

    for date, group in df.groupby("Date"):
        group = group.sort_values(pred_col).copy()
        n = len(group)

        if n < n_buckets:
            group["bucket"] = np.nan
        else:
            ranks = group[pred_col].rank(method="first")
            group["bucket"] = np.ceil(ranks / n * n_buckets).astype(int)
            group["bucket"] = group["bucket"].clip(1, n_buckets)

        out.append(group)

    return pd.concat(out, ignore_index=True)


In [ ]:
def compute_daily_long_short(df, bucket_col="bucket", realized_col="ret_next_day", n_buckets=10):
    """
    For each day:
    - long top bucket
    - short bottom bucket
    - compute long-short return
    """
    daily_ls = []

    for date, day_df in df.groupby("Date"):
        top = day_df[day_df[bucket_col] == n_buckets]
        bottom = day_df[day_df[bucket_col] == 1]

        if len(top) == 0 or len(bottom) == 0:
            continue

        long_ret = top[realized_col].mean()
        short_ret = bottom[realized_col].mean()
        ls_ret = long_ret - short_ret

        daily_ls.append({
            "Date": date,
            "long_ret": long_ret,
            "short_ret": short_ret,
            "long_short_ret": ls_ret,
            "n_long": len(top),
            "n_short": len(bottom),
        })

    return pd.DataFrame(daily_ls)

In [ ]:
def summarize_portfolio(df, ret_col="long_short_ret"):
    mean_ret = df[ret_col].mean()
    std_ret = df[ret_col].std()

    daily_sharpe = mean_ret / std_ret if std_ret > 0 else np.nan
    annualized_sharpe = daily_sharpe * np.sqrt(252) if std_ret > 0 else np.nan

    summary = {
        "mean_daily_return": mean_ret,
        "std_daily_return": std_ret,
        "daily_sharpe": daily_sharpe,
        "annualized_sharpe": annualized_sharpe,
        "n_days": len(df),
    }
    return pd.Series(summary)

In [ ]:
def run_downstream_pipeline_for_model(embedding_df, returns_df, eligible_df, cfg, alpha=1.0):
    regression_df = embedding_df.merge(
        returns_df,
        on=["Date", "Stock_symbol"],
        how="inner"
    )

    regression_df = regression_df.merge(
        eligible_df,
        on=["Date", "Stock_symbol"],
        how="inner"
    )

    regression_df = regression_df.dropna(subset=["ret_next_day"]).copy()
    regression_df["month"] = pd.to_datetime(regression_df["Date"]).dt.to_period("M").astype(str)

    embedding_cols = [c for c in regression_df.columns if c.startswith("emb_")]

    monthly_models = fit_monthly_ridge_models(
        regression_df,
        embedding_cols=embedding_cols,
        target_col="ret_next_day",
        alpha=alpha,
    )

    pred_df = build_oos_predictions(
        regression_df,
        embedding_cols=embedding_cols,
        monthly_models=monthly_models,
        target_col="ret_next_day",
    )

    n_buckets = 5 if cfg.debug else 10
    pred_df = assign_daily_buckets(
        pred_df,
        pred_col="pred_ret",
        n_buckets=n_buckets
    )

    portfolio_df = compute_daily_long_short(
        pred_df,
        bucket_col="bucket",
        realized_col="ret_next_day",
        n_buckets=n_buckets
    )

    summary = summarize_portfolio(portfolio_df)

    return {
        "regression_df": regression_df,
        "monthly_models": monthly_models,
        "pred_df": pred_df,
        "portfolio_df": portfolio_df,
        "summary": summary,
    }

In [ ]:
all_results = {}
all_summaries = []

for model_name, emb_df in embedding_tables.items():
    print(f"\nRunning downstream pipeline for {model_name}...")
    
    res = run_downstream_pipeline_for_model(
        embedding_df=emb_df,
        returns_df=returns_df,
        eligible_df=eligible_df,
        cfg=cfg,
        alpha=1.0,
    )
    
    all_results[model_name] = res

    summary_row = res["summary"].to_dict()
    summary_row["model"] = model_name
    all_summaries.append(summary_row)

summary_df = pd.DataFrame(all_summaries)
summary_df = summary_df[
    ["model", "mean_daily_return", "std_daily_return", "daily_sharpe", "annualized_sharpe", "n_days"]
].sort_values("annualized_sharpe", ascending=False)

summary_df

In [ ]:
summary_path = os.path.join(cfg.output_dir, "summary_df.csv")
summary_df.to_csv(summary_path, index=False)
print("Saved summary:", summary_path)

In [ ]:
plt.figure(figsize=(12, 6))

for model_name, res in all_results.items():
    portfolio_df = res["portfolio_df"].copy().sort_values("Date")
    portfolio_df["cum_long_short"] = portfolio_df["long_short_ret"].cumsum()

    plt.plot(
        pd.to_datetime(portfolio_df["Date"]),
        portfolio_df["cum_long_short"],
        label=model_name
    )

plt.xlabel("Date")
plt.ylabel("Cumulative return")
plt.title("Long-Short Portfolio Cumulative Return by Model")
plt.legend()
plt.grid(True)

plot_path = os.path.join(cfg.output_dir, "model_comparison_plot.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print("Saved plot:", plot_path)

plt.show()

In [ ]:
def run_repeated_subsample_pipeline(
    embedding_tables,
    returns_df,
    eligible_df,
    cfg,
    rows_per_year,
    n_repeats=5,
    reference_model="rolling_chronobert",
    base_random_state=40,
    alpha=1.0,
    fill_missing_daily_ret=0.0,
):
    """
    Repeatedly subsample rows_per_year from the CURRENT saved embedding tables,
    run the downstream pipeline for each model, and aggregate results.

    Parameters
    ----------
    embedding_tables : dict[str, pd.DataFrame]
        Already-loaded embedding tables.
    returns_df : pd.DataFrame
    eligible_df : pd.DataFrame
    cfg : Config
    rows_per_year : int
        Number of stock-day rows per year to sample from the saved embeddings.
    n_repeats : int
        Number of repeated subsamples.
    reference_model : str
        Model whose embedding table is used to define the sampled (Date, Stock_symbol) keys.
    base_random_state : int
        Seed base; repeat r uses base_random_state + r.
    alpha : float
        Ridge alpha.
    fill_missing_daily_ret : float
        Value used when a repeat/model has no portfolio return on a date in the master calendar.

    Returns
    -------
    dict with:
        repeat_level_summary_df
        model_summary_df
        avg_cumret_by_model
        all_repeat_results
    """

    if reference_model not in embedding_tables:
        raise ValueError(f"reference_model '{reference_model}' not found in embedding_tables")

    reference_df = embedding_tables[reference_model].copy()
    reference_df["year"] = pd.to_datetime(reference_df["Date"]).dt.year

    # Master trading calendar for plotting repeated cumulative returns
    master_dates = pd.Series(pd.to_datetime(returns_df["Date"]).drop_duplicates().sort_values().unique())
    master_dates.name = "Date"

    all_repeat_results = []
    repeat_level_rows = []
    plot_series_rows = []

    for repeat_idx in range(n_repeats):
        random_state = base_random_state + repeat_idx

        sampled_keys = (
            reference_df
            .groupby("year", group_keys=False)
            .apply(lambda x: x.sample(min(len(x), rows_per_year), random_state=random_state))
            .reset_index(drop=True)[["Date", "Stock_symbol"]]
            .drop_duplicates()
        )

        print(f"\n{'='*80}")
        print(f"Repeat {repeat_idx + 1}/{n_repeats}")
        print(f"Random state: {random_state}")
        print(f"Sampled keys: {len(sampled_keys)}")

        sampled_embedding_tables = {}
        for model_name, emb_df in embedding_tables.items():
            sampled_embedding_tables[model_name] = emb_df.merge(
                sampled_keys,
                on=["Date", "Stock_symbol"],
                how="inner"
            ).copy()

            print(model_name, sampled_embedding_tables[model_name].shape)

        repeat_result = {"repeat_idx": repeat_idx, "random_state": random_state, "models": {}}

        for model_name, emb_df in sampled_embedding_tables.items():
            print(f"\nRunning downstream pipeline for {model_name} (repeat {repeat_idx + 1})...")

            res = run_downstream_pipeline_for_model(
                embedding_df=emb_df,
                returns_df=returns_df,
                eligible_df=eligible_df,
                cfg=cfg,
                alpha=alpha,
            )

            repeat_result["models"][model_name] = res

            # Repeat-level summary
            summary_row = res["summary"].to_dict()
            summary_row["model"] = model_name
            summary_row["repeat_idx"] = repeat_idx
            summary_row["random_state"] = random_state
            summary_row["rows_per_year"] = rows_per_year
            repeat_level_rows.append(summary_row)

            # Build repeat-level cumulative return series on common date axis
            portfolio_df = res["portfolio_df"].copy()
            portfolio_df["Date"] = pd.to_datetime(portfolio_df["Date"])

            daily_ret = (
                portfolio_df[["Date", "long_short_ret"]]
                .drop_duplicates(subset=["Date"])
                .set_index("Date")
                .reindex(master_dates)
            )

            daily_ret["long_short_ret"] = daily_ret["long_short_ret"].fillna(fill_missing_daily_ret)
            daily_ret["cum_long_short"] = daily_ret["long_short_ret"].cumsum()
            daily_ret = daily_ret.reset_index()

            tmp_plot = daily_ret.copy()
            tmp_plot["model"] = model_name
            tmp_plot["repeat_idx"] = repeat_idx
            plot_series_rows.append(tmp_plot)

        all_repeat_results.append(repeat_result)

    repeat_level_summary_df = pd.DataFrame(repeat_level_rows)

    plot_df = pd.concat(plot_series_rows, ignore_index=True)

    # Aggregate cumulative return paths across repeats
    avg_cumret_by_model = (
        plot_df.groupby(["model", "Date"])
        .agg(
            mean_cum_long_short=("cum_long_short", "mean"),
            std_cum_long_short=("cum_long_short", "std"),
            n_repeats=("cum_long_short", "count"),
        )
        .reset_index()
    )
    avg_cumret_by_model["std_cum_long_short"] = avg_cumret_by_model["std_cum_long_short"].fillna(0.0)

    # Repeat-level metric aggregation
    model_summary_df = (
        repeat_level_summary_df.groupby("model")
        .agg(
            mean_mean_daily_return=("mean_daily_return", "mean"),
            std_mean_daily_return=("mean_daily_return", "std"),
            mean_std_daily_return=("std_daily_return", "mean"),
            std_std_daily_return=("std_daily_return", "std"),
            mean_daily_sharpe=("daily_sharpe", "mean"),
            std_daily_sharpe=("daily_sharpe", "std"),
            mean_annualized_sharpe=("annualized_sharpe", "mean"),
            std_annualized_sharpe=("annualized_sharpe", "std"),
            mean_n_days=("n_days", "mean"),
        )
        .reset_index()
        .sort_values("mean_annualized_sharpe", ascending=False)
    )

    return {
        "repeat_level_summary_df": repeat_level_summary_df,
        "model_summary_df": model_summary_df,
        "avg_cumret_by_model": avg_cumret_by_model,
        "all_repeat_results": all_repeat_results,
    }


def plot_repeated_subsample_results(
    avg_cumret_by_model,
    model_summary_df,
    band="std",
    title=None,
):
    """
    band:
        'std'  -> mean ± 1 std across repeats
        'sem'  -> mean ± 1 standard error across repeats
        None   -> no band
    """

    plt.figure(figsize=(12, 6))

    for model_name in model_summary_df["model"]:
        tmp = avg_cumret_by_model[avg_cumret_by_model["model"] == model_name].copy()
        tmp = tmp.sort_values("Date")

        x = tmp["Date"]
        y = tmp["mean_cum_long_short"].values
        s = tmp["std_cum_long_short"].values
        n = tmp["n_repeats"].values

        plt.plot(x, y, label=model_name)

        if band == "std":
            lower = y - s
            upper = y + s
            plt.fill_between(x, lower, upper, alpha=0.15)
        elif band == "sem":
            sem = np.divide(s, np.sqrt(np.maximum(n, 1)))
            lower = y - sem
            upper = y + sem
            plt.fill_between(x, lower, upper, alpha=0.15)

    plt.xlabel("Date")
    plt.ylabel("Average cumulative long-short return")
    if title is None:
        title = "Average cumulative long-short return across repeated subsamples"
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:

repeated_out = run_repeated_subsample_pipeline(
    embedding_tables=embedding_tables,
    returns_df=returns_df,
    eligible_df=eligible_df,
    cfg=cfg,
    rows_per_year=5000,   
    n_repeats=5,          
    reference_model="rolling_chronobert",
    base_random_state=40,
    alpha=1,
    fill_missing_daily_ret=0.0,
)

repeated_out["model_summary_df"]

In [ ]:

plot_repeated_subsample_results(
    avg_cumret_by_model=repeated_out["avg_cumret_by_model"],
    model_summary_df=repeated_out["model_summary_df"],
    band="std",   # or "sem"
    title="Finance pipeline: repeated subsamples from saved embedding matrices",
)


repeated_out["repeat_level_summary_df"].sort_values(
    ["model", "repeat_idx"]
)[["model", "repeat_idx", "annualized_sharpe", "mean_daily_return", "n_days"]]